In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
#import the data for 2017-2018

alq_18 = pd.read_sas("ALQ_J.XPT")
demo_18 = pd.read_sas("DEMO_J.XPT")
dpq_18 = pd.read_sas("DPQ_J.XPT")
duq_18 = pd.read_sas("DUQ_J.XPT")
mcq_18 = pd.read_sas("MCQ_J.XPT")
slq_18 = pd.read_sas("SLQ_J.XPT")
smq_18 = pd.read_sas("SMQ_J.XPT")

In [ ]:
#make sure each separate imported data set has the "SEQN" column for joining

for name, df in {
    "demo": demo_18,
    "dpq": dpq_18,
    "alq": alq_18,
    "duq": duq_18,
    "mcq": mcq_18,
    "slq": slq_18,
    "smq": smq_18
}.items():
    print(name, "SEQN" in df.columns)

In [ ]:
#make a copy of the "demo" data table for a template
data = demo_18.copy()

#merge all of the data, left join on "SEQN" to make one table for 2017-2018
data = data.merge(dpq_18, on="SEQN", how="left")
data = data.merge(alq_18, on="SEQN", how="left")
data = data.merge(duq_18, on="SEQN", how="left")
data = data.merge(mcq_18, on="SEQN", how="left")
data = data.merge(slq_18, on="SEQN", how="left")
data = data.merge(smq_18, on="SEQN", how="left")

In [ ]:
#check number of rows and columns
data.shape

In [ ]:
#look at the head
data.head()

In [ ]:
#look for missing data
data.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
#find the PHQ-9 (Patient Health Questionnaire-9) variables
data.filter(like="DPQ").columns

In [ ]:
#create depression severity score
phq_items = [
    "DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050", "DPQ060", "DPQ070",
       "DPQ080", "DPQ090"
]

#for every PHQ symptom question replace response codes 7 & 9 with missing values
#in the NHANES data 7 = refused to answer & 9 = don't know
for col in phq_items:
    data[col] = data[col].replace({7: np.nan, 9: np.nan})

#calculate each patient's total PHQ-9 score by adding together responses across the PHQ items
data["PHQ9_TOTAL"] = data[phq_items].sum(axis=1, skipna=True)

In [ ]:
#look at distribution
data["PHQ9_TOTAL"].describe()

In [ ]:
#round the PHQ-9 score to the nearest whole number
data["PHQ9_TOTAL"] = data["PHQ9_TOTAL"].round(0)

In [ ]:
#look at distribution histogram
data["PHQ9_TOTAL"].hist()

In [ ]:
#create a binary depression outcome greater or equal to 10 (threshold for moderate depression)
data["DEPRESSED"] = (data["PHQ9_TOTAL"] >= 10).astype(int)

In [ ]:
#check class balance
data["DEPRESSED"].value_counts()

In [ ]:
#percentage of patients who are depressed vs not depressed
data["DEPRESSED"].value_counts(normalize=True)

In [ ]:
#create new column called "Function_Impairement" & put values from DPQ100 in it
data["FUNCTION_IMPAIRMENT"] = data["DPQ100"]

In [ ]:
#use just age and sex as variables
features = [
    "RIDAGEYR", #age
    "RIAGENDR", #sex
]

#create new dataset with just features above & remove rows with missing values
model_df = data[features + ["DEPRESSED"]].dropna()

print(model_df.shape)

In [ ]:
#create X (features) and y (target) variables
X = model_df[features]
y = model_df["DEPRESSED"]

In [ ]:
#look at predictor variables
X.describe()

In [ ]:
#look at head of predictor variables
X.head()

In [ ]:
#import train_test_split from sklearn
from sklearn.model_selection import train_test_split

#create X (features) & y (target) variables
X = model_df[features]
y = model_df["DEPRESSED"]

#split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
#check class balances
y_train.value_counts(normalize=True)

In [ ]:
#check class balances
y_test.value_counts(normalize=True)

In [ ]:
#rows & columns check
X_train.shape

In [ ]:
#rows & columns check
X_test.shape

In [ ]:
#import LogisticRegression from sklearn
from sklearn.linear_model import LogisticRegression

#build a LogisticRegression model
lm_model = LogisticRegression(max_iter=1000)

#fit the LogisticRegression model
lm_model.fit(X_train, y_train)

In [ ]:
#import evaluation metrics from sklearn
from sklearn.metrics import roc_auc_score, classification_report

#evaluate model performance
lm_pred = lm_model.predict_proba(X_test)[:,1]

#get AUROC
roc_auc_score(y_test, lm_pred)

In [ ]:
#Identify sleep variables
slq_18.columns

In [ ]:
#explore the SLQ320 column
data["SLQ320"].head()

In [ ]:
#explore the SLQ320 column
data["SLQ320"].unique()

In [ ]:
#keep sleep duration, daytime fatigue and sleep medication use
sleep_features = [
    "SLD012", #sleep duration
    "SLQ040", #daytime sleepiness
    "SLQ120"  # sleep medication
]

In [ ]:
#add age, gender & sleep features to features list
features = [
    "RIDAGEYR", #age
    "RIAGENDR"  #sex
] + sleep_features #sleep features above

In [ ]:
#build a modeling dataset adding target variable "depressed"
model_df = data[features + ["DEPRESSED"]].dropna()

In [ ]:
#check columns and rows
model_df.shape

In [ ]:
#import train_test_split from sklearn
from sklearn.model_selection import train_test_split

#build features and target variables
X = model_df[features]
y = model_df["DEPRESSED"]

#split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
#look at what type of data is in X_train
X_train.dtypes

In [ ]:
#import LogisticRegression model from sklearn
from sklearn.linear_model import LogisticRegression

#build a Logistic Regression model
lm_model2 = LogisticRegression(max_iter=1000)

#fit the Logistic Regression model
lm_model2.fit(X_train, y_train)

In [ ]:
#import evaluation metrics from sklearn
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

#get probabilities and roc score
lm_pred2 = lm_model2.predict_proba(X_test)[:,1]
auc_sleep = roc_auc_score(y_test, lm_pred2)

#print roc score
auc_sleep

In [ ]:
#convert probabilities into 0 or 1
lm_pred2 = (lm_pred2 >= 0.5).astype(int)

#build confusion matrix
confusion_matrix(y_test, lm_pred2)

In [ ]:
#print a classification report
print(classification_report(y_test, lm_pred2))

In [ ]:
#build a logistic coefficients list & sort from lowest to highest
coef_df = pd.Series(
    lm_model2.coef_[0], #predicting class "1" (DEPRESSED)
    index=features #label
).sort_values() #sort

#print the logistic coefficients list, higher the value/higher probability of depression
coef_df

In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

#generate a ROC Curve Display and save to folder
RocCurveDisplay.from_estimator(lm_model2, X_test, y_test)
plt.savefig("roc_curve.png", dpi=300)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# If you already have predicted probabilities:
lm_pred2 = (lm_pred2 >= 0.5).astype(int)

#assign variables & labels
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, lm_pred2,
    display_labels=["Not depressed", "Depressed"],
    values_format="d"
)

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

num_features = features  #numeric features

#preprocess data pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_features)
    ],
    remainder="drop"
)

#add preprocess pipeline to model
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

#show pipeline
clf

In [ ]:
#GRABBED THE PRESCRIPTION DATA, CHECK IT OUT
df_rx = pd.read_sas("RXQ_RX_J.XPT")
df_rx.head()

In [ ]:
#make prescriptions readable
df_rx['RXDDRUG'] = df_rx['RXDDRUG'].str.decode('utf-8')
df_rx['RXDDRUG'] = df_rx['RXDDRUG'].str.upper()

In [ ]:
#identify the antidepressants
def classify_antidepressant(drug):
    if pd.isna(drug):
        return None

    drug = drug.upper()

    ssri = ['FLUOXETINE','SERTRALINE','CITALOPRAM','ESCITALOPRAM','PAROXETINE','FLUVOXAMINE']
    snri = ['VENLAFAXINE','DESVENLAFAXINE','DULOXETINE','LEVOMILNACIPRAN']
    tca = ['AMITRIPTYLINE','NORTRIPTYLINE','IMIPRAMINE','DESIPRAMINE','CLOMIPRAMINE','DOXEPIN']
    atypical = ['BUPROPION','MIRTAZAPINE','TRAZODONE','VILAZODONE','VORTIOXETINE']
    
    if any(d in drug for d in ssri):
        return 'SSRI'
    if any(d in drug for d in snri):
        return 'SNRI'
    if any(d in drug for d in tca):
        return 'TCA'
    if any(d in drug for d in atypical):
        return 'ATYPICAL'
    
    return None

df_rx['AD_CLASS'] = df_rx['RXDDRUG'].apply(classify_antidepressant)

In [ ]:
#keep only the antidepressants
df_ad = df_rx[df_rx['AD_CLASS'].notnull()]

In [ ]:
#only keep people taking one antidepressant
ad_counts = df_ad.groupby('SEQN')['AD_CLASS'].nunique().reset_index()
ad_counts = ad_counts[ad_counts['AD_CLASS'] == 1]

single_class_ids = ad_counts['SEQN']

df_ad_single = df_ad[df_ad['SEQN'].isin(single_class_ids)]

df_treatment = df_ad_single[['SEQN','AD_CLASS']].drop_duplicates()

In [ ]:
#merge prescriptions with main cdc dataset
data = data.merge(df_treatment, on='SEQN', how='inner')

data.head()

In [ ]:
#show column labels
df_treatment.columns

In [ ]:
#show first five
df_treatment.head()

In [ ]:
#make seqn types match (integers)
data['SEQN'] = data['SEQN'].astype(int)
df_treatment['SEQN'] = df_treatment['SEQN'].astype(int)

In [ ]:
#merge treatment class into the main dataset
data = data.merge(df_treatment, on='SEQN', how='inner')

In [ ]:
#confirm the column exists
print(data.columns)
print(data.shape)

# Case 1: already have AD_CLASS
if "AD_CLASS" in data.columns:
    pass

# Case 2: merge-created columns exist, rebuild AD_CLASS from them
elif "AD_CLASS_x" in data.columns or "AD_CLASS_y" in data.columns:
    if "AD_CLASS_x" in data.columns and "AD_CLASS_y" in data.columns:
        data["AD_CLASS"] = data["AD_CLASS_x"].combine_first(data["AD_CLASS_y"])
    elif "AD_CLASS_x" in data.columns:
        data["AD_CLASS"] = data["AD_CLASS_x"]
    elif "AD_CLASS_y" in data.columns:
        data["AD_CLASS"] = data["AD_CLASS_y"]

# Case 3: no AD_CLASS at all, merge df_treatment again
else:
    data["SEQN"] = data["SEQN"].astype(int)
    df_treatment["SEQN"] = df_treatment["SEQN"].astype(int)
    data = data.merge(df_treatment, on="SEQN", how="inner")

# Clean up duplicate columns if they exist
cols_to_drop = [c for c in ["AD_CLASS_x", "AD_CLASS_y"] if c in data.columns]
if len(cols_to_drop) > 0:
    data = data.drop(columns=cols_to_drop)

print("Final AD columns:", [c for c in data.columns if "AD_CLASS" in c])
print(data["AD_CLASS"].value_counts())
print(data.shape)


In [ ]:
# Keep the final AD_CLASS and drop merge leftovers
cols_to_drop = [c for c in ["AD_CLASS_x", "AD_CLASS_y"] if c in data.columns]
data = data.drop(columns=cols_to_drop)

# Sanity check
data[["SEQN", "AD_CLASS"]].head(), data["AD_CLASS"].value_counts()

In [ ]:
#map treatment class labels
treat_map = {"ATYPICAL": 0, "SNRI": 1, "SSRI": 2, "TCA": 3}

#replace string labels with mapped labels
data["treatment"] = data["AD_CLASS"].map(treat_map).astype(int)

In [ ]:
#assign outcome variable as binary 1/0
Y = data["DEPRESSED"].astype(int).values

#assign treatment variable as mapped values
T = data["treatment"].values

In [ ]:
#name predictors
X_cols=["RIDAGEYR", "RIAGENDR", "RIDRETH1"]

#select only predictor columns from data
X = data[X_cols].copy()

In [ ]:
#Handle missing rows simply
model_df = data[["DEPRESSED", "treatment"] + X_cols].dropna().copy()

In [ ]:
#convert DEPRESSED column values to integers 1/0
Y = model_df["DEPRESSED"].astype(int).values

#make sure treatment values are integers
T = model_df["treatment"].astype(int).values

#select only columns listed in X_cols
X = model_df[X_cols].values

In [ ]:
#check rows/columns & count each treatment sample
model_df.shape,pd.Series(T).value_counts()

In [ ]:
##Going to remove TCA for cleaner estimates and lower variance
data = data[data["AD_CLASS"] != "TCA"].copy()

#Check new counts
data["AD_CLASS"].value_counts(), data.shape

In [ ]:
#Remap the treatment codes
treat_map = {"ATYPICAL": 0, "SNRI": 1, "SSRI": 2}
data["treatment"] = data["AD_CLASS"].map(treat_map).astype(int)

#check counts
data["treatment"].value_counts()

In [ ]:
#Define Outcome Y
Y = data["DEPRESSED"].astype(int).values

#define treatment T
T = data["treatment"].values

In [ ]:
#Define Covariates (start simple)
X_cols = ["RIDAGEYR", "RIAGENDR", "RIDRETH1"]

#only select assigned columns & name model_df
model_df = data[["DEPRESSED", "treatment"] + X_cols].dropna().copy()

#make sure value types are correct for Y,T & X
Y = model_df["DEPRESSED"].astype(int).values
T = model_df["treatment"].astype(int).values
X = model_df[X_cols].values

#check model_df rows/columns
model_df.shape

In [ ]:
#update drlearner categories
categories=[0, 1, 2]

In [ ]:
import econml
print(econml.__version__)

In [ ]:
from econml.dr import DRLearner
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, XGBRegressor

#create regression model
model_regression = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=22
)

#create propensity model
model_propensity = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=22
)

#create dr_learner model
dr_learner = DRLearner(
    model_regression=model_regression,
    model_propensity=model_propensity,
    discrete_outcome=True,
    categories=[0,1,2],
    random_state=22
)

In [ ]:
#check how often combinations of values occur
pd.crosstab(data["AD_CLASS"], data["DEPRESSED"], normalize="index")

In [ ]:
#checks & balances

#Depression rate by treatment class (raw, unadjusted)
print(pd.crosstab(data["AD_CLASS"], data["DEPRESSED"], normalize="index"))

In [ ]:
#counts by class
print(data["AD_CLASS"].value_counts())

In [ ]:
#show modeling frame
print("Model rows:", model_df.shape[0])
print("Treatment counts:", pd.Series(T).value_counts().sort_index().to_dict())
print("Outcome mean (depressed rate):", Y.mean())

In [ ]:
#combined stratification label keeps balance on BOTH T and Y
strata = model_df["treatment"].astype(str) + "_" + model_df["DEPRESSED"].astype(str)

#20% final holdout
idx_all = np.arange(len(model_df))
idx_model, idx_holdout = train_test_split(
    idx_all,
    test_size=0.20,
    random_state=22,
    stratify=strata
)

#split the 80% into train/val (70/30)
strata_model = strata.iloc[idx_model]
idx_train, idx_val = train_test_split(
    idx_model,
    test_size=0.30,
    random_state=22,
    stratify=strata_model
)

#helper for subsets of X,T & Y
def slice_arrays(idxs):
    return X[idxs], T[idxs], Y[idxs]

#train, val, & test split
X_train, T_train, Y_train = slice_arrays(idx_train)
X_val, T_val, Y_val = slice_arrays(idx_val)
X_hold, T_hold, Y_hold = slice_arrays(idx_holdout)

#checks & balances
print("Train/Val/Hold sizes:",
      len(idx_train),
      len(idx_val),
      len(idx_holdout))
print("Train depressed rate:", Y_train.mean(),
      "Val:", Y_val.mean(),
      "Hold:", Y_hold.mean())
print("Train T dist:",
      pd.Series(T_train).value_counts(normalize=True).sort_index().to_dict())
print("Val T dist:",
      pd.Series(T_val).value_counts(normalize=True).sort_index().to_dict())
print("Hold T dist:",
      pd.Series(T_hold).value_counts(normalize=True).sort_index().to_dict())

In [ ]:
#outcome model: predicts P(DEPRESSED | X,T)
model_regression = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=22,
    eval_metric="logloss"
)

#propensity model: predicts P(T|X) for 3 classes
model_propensity = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=22,
    eval_metric="mlogloss"
)


#causal model
dr = DRLearner(
    model_regression=model_regression, #outcome
    model_propensity=model_propensity, #propensity
    discrete_outcome=True,
    categories=[0, 1, 2], #0=ATYPICAL, 1=SNRI, 2=SSRI
    random_state=22,
)

#fit the model
dr.fit(Y_train, T_train, X=X_train)
print("DRLearner fit complete.")

In [ ]:
marginal = dr.const_marginal_effect(X_val) #often shape (n, 1, k) for discrete outcome
marginal2 = marginal[:, 0, :] #shape (n, 2)

print("marginal2 shape:", marginal2.shape)
print("categories:", dr.categories)

In [ ]:
# Recreate train/val tables from model_df using your saved indices
train_df = model_df.iloc[idx_train].copy()
val_df = model_df.iloc[idx_val].copy()

# Features for the standalone outcome model:
# covariates + treatment
outcome_features = ["RIDAGEYR", "RIAGENDR", "RIDRETH1", "treatment"]

#prepare training dataset for outcome model
X_train_outcome = train_df[outcome_features]
y_train_outcome = train_df["DEPRESSED"].astype(int)

#prepare validation dataset for outcome model
X_val_outcome = val_df[outcome_features]
y_val_outcome = val_df["DEPRESSED"].astype(int)

In [ ]:
#fit a standalone outcome model
outcome_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytress=0.8,
    random_state=22,
    eval_metric="logloss"
)

#fit the model
outcome_model.fit(X_train_outcome, y_train_outcome)

In [ ]:
#get baseline risk under ATYPICAL
X_val_t0 = X_val_outcome.copy()
X_val_t0["treatment"] = 0 #ATYPICAL baseline

#estimate potential outcomes under ATYPICAL 
p_atypical = outcome_model.predict_proba(X_val_t0)[:,1]

print(p_atypical[:10])
print(p_atypical.shape)

In [ ]:
#convert treatment effects into risks
effect_snri = marginal2[:, 0]
effect_ssri = marginal2[:,1]

#construct counterfactual outcomes
risk_atypical = p_atypical
risk_snri = np.clip(p_atypical + effect_snri, 0, 1)
risk_ssri = np.clip(p_atypical + effect_ssri, 0, 1)

In [ ]:
#put it into a dataframe
risk_df = pd.DataFrame({
    "risk_ATYPICAL": risk_atypical,
    "risk_SNRI": risk_snri,
    "risk_SSRI": risk_ssri,
})

#check
risk_df.head()

In [ ]:
#check
risk_df.describe()

In [ ]:
#Red flag because it looks like many predicted risks were pushed above 1
#then clipped. Might be because treatment effects are being added
#incorrectly
#DRLearner marginal effects represent risk differences, but when the 
#outcome model is a classifier they are acutally on the log-odds scale
#not the probability scale so adding them directly to probabilities
#creates nonsense numbers

#going to predict the probability of depression under each treatment
#separately instead
X_val_t0 = X_val_outcome.copy()
X_val_t1 = X_val_outcome.copy()
X_val_t2 = X_val_outcome.copy()

#counterfactual datasets
X_val_t0["treatment"] = 0   # ATYPICAL
X_val_t1["treatment"] = 1   # SNRI
X_val_t2["treatment"] = 2   # SSRI

In [ ]:
#then predict risk under each treatment
risk_atypical = outcome_model.predict_proba(X_val_t0)[:,1]
risk_snri = outcome_model.predict_proba(X_val_t1)[:,1]
risk_ssri = outcome_model.predict_proba(X_val_t2)[:,1]

In [ ]:
#build the risk dataframe
risk_df = pd.DataFrame({
    "risk_ATYPICAL": risk_atypical,
    "risk_SNRI": risk_snri,
    "risk_SSRI": risk_ssri
})

#check
risk_df.describe()

In [ ]:
#build treatment recommendation system
risk_df["recommended_treatment"] = risk_df.idxmin(axis=1)

#check
risk_df.head()

In [ ]:
#check the recommendation distribution
risk_df["recommended_treatment"].value_counts(normalize=True)

In [ ]:
#predict depression probability for validation set
from sklearn.metrics import roc_auc_score, brier_score_loss

#evaluate the model
val_probs = outcome_model.predict_proba(X_val_outcome)[:,1]

print("Validation AUROC:", roc_auc_score(y_val_outcome, val_probs))
print("Validation Brier Score:", brier_score_loss(y_val_outcome, val_probs))

In [ ]:
#verify which variables are in data
for col in ["INDFMPIR","BMXBMI","SMQ020","ALQ101","BPQ020","DIQ010"]:
    print(col, col in data.columns)

In [ ]:
#not happy with the AUROC so going to add more features
X_cols = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH1",
    "INDFMPIR",
    "SMQ020"
]

#create dataframe with assigned columns
model_df = data[["DEPRESSED", "treatment"] + X_cols].dropna().copy()

#check
print("New model_df shape:", model_df.shape)


In [ ]:
# Create stratification label to preserve treatment + outcome balance
strata = model_df["treatment"].astype(str) + "_" + model_df["DEPRESSED"].astype(str)

# Create fresh indices from THIS model_df
idx_all = np.arange(len(model_df))

# 20% holdout
idx_model, idx_holdout = train_test_split(
    idx_all,
    test_size=0.20,
    random_state=22,
    stratify=strata
)

# 30% of remaining 80% becomes validation
strata_model = strata.iloc[idx_model]
idx_train, idx_val = train_test_split(
    idx_model,
    test_size=0.30,
    random_state=22,
    stratify=strata_model
)

print("Train/Val/Hold sizes:", len(idx_train), len(idx_val), len(idx_holdout))

In [ ]:
#create training, validation and test set
train_df = model_df.iloc[idx_train].copy()
val_df   = model_df.iloc[idx_val].copy()
hold_df  = model_df.iloc[idx_holdout].copy()

#define outcome features
outcome_features = X_cols + ["treatment"]

#training
X_train_outcome = train_df[outcome_features]
y_train_outcome = train_df["DEPRESSED"].astype(int)

#validation
X_val_outcome = val_df[outcome_features]
y_val_outcome = val_df["DEPRESSED"].astype(int)

#test
X_hold_outcome = hold_df[outcome_features]
y_hold_outcome = hold_df["DEPRESSED"].astype(int)

In [ ]:
#XGBoost classifier uses gradient boosting for classification which will build
#sequential trees and improve performance by correcting previous errors
from xgboost import XGBClassifier

#create the model
outcome_model = XGBClassifier(
    n_estimators=300, #trees in model
    max_depth=4, #depth of tree
    learning_rate=0.05, #step size for boosting iteration
    subsample=0.8, #adds randomness
    colsample_bytree=0.8, #80% of features per tree
    random_state=22, #reproduce
    eval_metric="logloss" #to evaluate performance, good for binary
)

#train the model
outcome_model.fit(X_train_outcome, y_train_outcome)

#evaluate the model
val_probs = outcome_model.predict_proba(X_val_outcome)[:,1]

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss

#evaluate the model
print("Validation AUROC:", roc_auc_score(y_val_outcome, val_probs))
print("Validation Brier Score:", brier_score_loss(y_val_outcome, val_probs))

In [ ]:
#want to try again with more features
#load the asthma, arthritis and heart disease data
mcq_g = pd.read_sas("MCQ_G.XPT")
mcq_h = pd.read_sas("MCQ_H.XPT")
mcq_i = pd.read_sas("MCQ_I.XPT")
mcq_j = pd.read_sas("MCQ_J.XPT")

#concatenate together
mcq = pd.concat([mcq_g, mcq_h, mcq_i, mcq_j], ignore_index=True)

#check
mcq.head()

In [ ]:
#keep only the variables i need
mcq = mcq[[
    "SEQN",
    "MCQ010",   # asthma
    "MCQ160A",  # arthritis
    "MCQ160F"   # heart disease
]]

#check
mcq.head()

In [ ]:
#load the sleep data
slq_g = pd.read_sas("SLQ_G.XPT")
slq_h = pd.read_sas("SLQ_H.XPT")
slq_i = pd.read_sas("SLQ_I.XPT")
slq_j = pd.read_sas("SLQ_J.XPT")

#concatenate together
slq = pd.concat([slq_g, slq_h, slq_i, slq_j], ignore_index=True)

#subset teh data only keep ID and sleep duration
slq = slq[[
    "SEQN",
    "SLD010H"   # hours of sleep
]]

#check
slq.head()
data.head()

In [ ]:
#check for variables
[c for c in data.columns if c.startswith("MCQ") or c.startswith("SLD") or c.startswith("SLQ")]

In [ ]:
# Drop previously merged MCQ / sleep columns if they already exist
to_drop = [
    c for c in data.columns
    if c.startswith("MCQ") or c.startswith("SLD") or c.startswith("SLQ")
]

#remove uneccessary columns
data = data.drop(columns=to_drop, errors="ignore")

#check
print("After drop:", data.shape)

# Make sure SEQN types match
data["SEQN"] = data["SEQN"].astype(int)
mcq["SEQN"] = mcq["SEQN"].astype(int)
slq["SEQN"] = slq["SEQN"].astype(int)

# Merge once, cleanly
data = data.merge(mcq, on="SEQN", how="left")
data = data.merge(slq, on="SEQN", how="left")

#check
print("After clean merge:", data.shape)

In [ ]:
#resolve duplicates by combining them into a single column
if "SLD010H_x" in data.columns and "SLD010H_y" in data.columns:
    data["SLD010H"] = data["SLD010H_x"].combine_first(data["SLD010H_y"])
    data = data.drop(columns=["SLD010H_x", "SLD010H_y"])

In [ ]:
#assign feature columns
X_cols = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH1",
    "INDFMPIR",
    "SMQ020",
    "MCQ010",
    "MCQ160A",
    "MCQ160F",
    "SLD010H",
    "SLQ050"
]

#label codes that mean missing data
missing_codes = [7, 9, 77, 99, 777, 999, 7777, 9999]

#replace missing values with nan
for c in X_cols:
    if c in data.columns:
        data[c] = data[c].replace(missing_codes, np.nan)

#select all columns that start with SL
[c for c in data.columns if c.startswith("SL")]

In [ ]:
#name feature columns
candidate_X_cols = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH1",
    "INDFMPIR",
    "SMQ020",
    "MCQ010",
    "MCQ160A",
    "MCQ160F",
    "SLD010H"
]

#validate and filter featue list to match the dataset columns
X_cols = [c for c in candidate_X_cols if c in data.columns]

#check
print("Using X_cols:", X_cols)

In [ ]:
# Recode common NHANES missing-value codes
missing_codes = [7, 9, 77, 99, 777, 999, 7777, 9999]

#replace the missing codes with nan
for c in X_cols:
    data[c] = data[c].replace(missing_codes, np.nan)

#select assigned columns
cols = ["DEPRESSED", "treatment"] + X_cols

#check
print(data[cols].isna().sum().sort_values(ascending=False))

In [ ]:
#create dataframe with assigned columns
model_df = data[["DEPRESSED", "treatment"] + X_cols].dropna().copy()

#check
print("Rows remaining:", model_df.shape)

In [ ]:
#proportion of missing for each column from greatest to smallest
missing_pct = data[cols].isna().mean().sort_values(ascending=False)
print(missing_pct)

In [ ]:
#assign feature columns
X_cols = [
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH1",
    "INDFMPIR",
    "SMQ020",
    "MCQ010",
    "MCQ160A",
    "MCQ160F"
]

In [ ]:
#create dataset using assigned columns
model_df = data[["DEPRESSED", "treatment"] + X_cols].dropna().copy()

#check
print("Rows remaining:", model_df.shape)

In [ ]:
#create a combine group variable-concatenate treatment and outcome values
#for each row
strata = model_df["treatment"].astype(str) + "_" + model_df["DEPRESSED"].astype(str)

#create array of integer indices from 0 to number of rows in model_df-1 to represent
#all row positions in the dataset
idx_all = np.arange(len(model_df))

#split into a training and holdout set
idx_model, idx_holdout = train_test_split(
    idx_all,
    test_size=0.20,
    random_state=22,
    stratify=strata
)

#subset of strata with the training indices ids_model, so treatment-outcome
#group labels are only in training data
strata_model = strata.iloc[idx_model]

#split again into training and validation sets
idx_train, idx_val = train_test_split(
    idx_model,
    test_size=0.30,
    random_state=22,
    stratify=strata_model
)

In [ ]:
#select rows from model_df using indices in idx_train, create training/validation set
train_df = model_df.iloc[idx_train].copy()
val_df   = model_df.iloc[idx_val].copy()

#define features for outcome model
outcome_features = X_cols + ["treatment"]

#create feature matrix for training
X_train_outcome = train_df[outcome_features]
y_train_outcome = train_df["DEPRESSED"].astype(int)

#create feature matrix for validation
X_val_outcome = val_df[outcome_features]
y_val_outcome = val_df["DEPRESSED"].astype(int)

In [ ]:
#train the model
outcome_model.fit(X_train_outcome, y_train_outcome)

#evaluate the model
val_probs = outcome_model.predict_proba(X_val_outcome)[:,1]

In [ ]:
#print results
print("Validation AUROC:", roc_auc_score(y_val_outcome, val_probs))
print("Validation Brier Score:", brier_score_loss(y_val_outcome, val_probs))

In [ ]:
#list columns like these
[c for c in data.columns if c in ["SLD012","SLQ040","SLQ120"]]

In [ ]:
#load all sleep datasets
slq_g = pd.read_sas("SLQ_G.XPT")
slq_h = pd.read_sas("SLQ_H.XPT")
slq_i = pd.read_sas("SLQ_I.XPT")
slq_j = pd.read_sas("SLQ_J.XPT")

In [ ]:
#combine all the sleep datasets
slq = pd.concat([slq_g, slq_h, slq_i, slq_j], ignore_index=True)

In [ ]:
#check the sleep datasets
print(slq.shape)
print(slq.columns)

In [ ]:
#extract the sleep predictors
sleep_vars = ["SEQN", "SLD012", "SLQ040", "SLQ120"]

#select columns in sleep_vars from slq dataframe and store in slq_sleep
slq_sleep = slq[sleep_vars]

In [ ]:
#merge with main dataset
data = data.merge(slq_sleep, on="SEQN", how="left")

In [ ]:
#verify
[c for c in data.columns if c in ["SLD012","SLQ040","SLQ120"]]

In [ ]:
#build feature dataset
X_cols = [
"RIDAGEYR",
"RIAGENDR",
"RIDRETH1",
"INDFMPIR",
"SMQ020",
"MCQ010",
"MCQ160A",
"MCQ160F",
"SLD012",
"SLQ040",
"SLQ120"
]

In [ ]:
#rebuild the modeling dataset
missing_codes = [7,9,77,99,777,999]

#replace missing data with nan
model_df = data[["DEPRESSED","treatment"] + X_cols].replace(missing_codes, np.nan)

#drop rows missing info
model_df = model_df.dropna()

#check
print("Rows remaining:", model_df.shape)

In [ ]:
#list all rows starting with SL
[c for c in slq.columns if c.startswith("SL")]

In [ ]:
#check that all years are in dataset
data["SDDSRVYR"].value_counts().sort_index()

In [ ]:
#all four cycles have not been included in the data, only cycle 10
#so need to go back and grab all four cycles
alq_g = pd.read_sas("ALQ_G.XPT")
alq_h = pd.read_sas("ALQ_H.XPT")
alq_i = pd.read_sas("ALQ_I.XPT")
alq_j = pd.read_sas("ALQ_J.XPT")

demo_g = pd.read_sas("DEMO_G.XPT")
demo_h = pd.read_sas("DEMO_H.XPT")
demo_i = pd.read_sas("DEMO_I.XPT")
demo_j = pd.read_sas("DEMO_j.XPT")

dpq_g = pd.read_sas("DPQ_G.XPT")
dpq_h = pd.read_sas("DPQ_H.XPT")
dpq_i = pd.read_sas("DPQ_I.XPT")
dpq_j = pd.read_sas("DPQ_J.XPT")

duq_g = pd.read_sas("DUQ_G.XPT")
duq_h = pd.read_sas("DUQ_H.XPT")
duq_i = pd.read_sas("DUQ_I.XPT")
duq_j = pd.read_sas("DUQ_J.XPT")

mcq_g = pd.read_sas("MCQ_G.XPT")
mcq_h = pd.read_sas("MCQ_H.XPT")
mcq_i = pd.read_sas("MCQ_I.XPT")
mcq_j = pd.read_sas("MCQ_J.XPT")

rxq_g = pd.read_sas("RXQ_RX_G.XPT")
rxq_h = pd.read_sas("RXQ_RX_H.XPT")
rxq_i = pd.read_sas("RXQ_RX_I.XPT")
rxq_j = pd.read_sas("RXQ_RX_J.XPT")

slq_g = pd.read_sas("SLQ_G.XPT")
slq_h = pd.read_sas("SLQ_H.XPT")
slq_i = pd.read_sas("SLQ_I.XPT")
slq_j = pd.read_sas("SLQ_J.XPT")

smq_g = pd.read_sas("SMQ_G.XPT")
smq_h = pd.read_sas("SMQ_H.XPT")
smq_i = pd.read_sas("SMQ_I.XPT")
smq_j = pd.read_sas("SMQ_J.XPT")

In [ ]:
#combine each 4 cycle group
alq = pd.concat([alq_g, alq_h, alq_i, alq_j], ignore_index=True)

demo = pd.concat([demo_g, demo_h, demo_i, demo_j], ignore_index=True)

dpq = pd.concat([dpq_g, dpq_h, dpq_i, dpq_j], ignore_index=True)

slq = pd.concat([slq_g, slq_h, slq_i, slq_j], ignore_index=True)

duq = pd.concat([duq_g, duq_h, duq_i, duq_j], ignore_index=True)

mcq = pd.concat([mcq_g, mcq_h, mcq_i, mcq_j], ignore_index=True)

rxq = pd.concat([rxq_g, rxq_h, rxq_i, rxq_j], ignore_index=True)

smq = pd.concat([smq_g, smq_h, smq_i, smq_j], ignore_index=True)

In [ ]:
#merge the datasets
data = demo.merge(alq, on="SEQN", how="inner")

data = data.merge(dpq, on="SEQN", how="left")
data = data.merge(slq, on="SEQN", how="left")
data = data.merge(duq, on="SEQN", how="left")
data = data.merge(mcq, on="SEQN", how="left")
data = data.merge(rxq, on="SEQN", how="left")
data = data.merge(smq, on="SEQN", how="left")

print(data.shape)

In [ ]:
#confirm the cycles
data["SDDSRVYR"].value_counts().sort_index()

#check
print(data.shape)

In [ ]:
#create depression outcome
phq_cols = [
"DPQ010","DPQ020","DPQ030","DPQ040",
"DPQ050","DPQ060","DPQ070","DPQ080","DPQ090"
]

#select all columns in phq_cols & sum row wise to get total depression score
data["PHQ9_TOTAL"] = data[phq_cols].sum(axis=1)

#check if score is under/equal to 10 & convert to T/F-1/0-likely depressed,
#not depressed
data["DEPRESSED"] = (data["PHQ9_TOTAL"] >= 10).astype(int)

In [ ]:
#feature engineer sleep
data["short_sleep"] = (data["SLD012"] < 6).astype(int)

#check if value over/equal to 2 and convert to 1=true, 0=false
data["sleep_problem"] = (data["SLQ040"] >= 2).astype(int)

#check if equal to 1, convert to 1=yes, 0 = no
data["sleep_med"] = (data["SLQ120"] == 1).astype(int)

In [ ]:
#define predictors
X_cols = [
"RIDAGEYR",
"RIAGENDR",
"RIDRETH1",
"INDFMPIR",
"SMQ020",
"MCQ010",
"MCQ160A",
"MCQ160F",
"short_sleep",
"sleep_problem",
"sleep_med"
]

In [ ]:
#rebuild the treatment table from rxq
# Decode drug names
rxq["RXDDRUG"] = rxq["RXDDRUG"].str.decode("utf-8")
rxq["RXDDRUG"] = rxq["RXDDRUG"].str.upper()

#create function to classify antidepressant classes
def classify_antidepressant(drug):
    if pd.isna(drug):
        return None

    ssri = ["FLUOXETINE", "SERTRALINE", "CITALOPRAM", "ESCITALOPRAM", "PAROXETINE", "FLUVOXAMINE"]
    snri = ["VENLAFAXINE", "DESVENLAFAXINE", "DULOXETINE", "LEVOMILNACIPRAN"]
    tca = ["AMITRIPTYLINE", "NORTRIPTYLINE", "IMIPRAMINE", "DESIPRAMINE", "CLOMIPRAMINE", "DOXEPIN"]
    atypical = ["BUPROPION", "MIRTAZAPINE", "TRAZODONE", "VILAZODONE", "VORTIOXETINE"]

    if any(d in drug for d in ssri):
        return "SSRI"
    if any(d in drug for d in snri):
        return "SNRI"
    if any(d in drug for d in tca):
        return "TCA"
    if any(d in drug for d in atypical):
        return "ATYPICAL"

    return None

#create new variable by classifying each drug into an antidepressant category
rxq["AD_CLASS"] = rxq["RXDDRUG"].apply(classify_antidepressant)

In [ ]:
#keep only the antidepressant users
df_ad = rxq[rxq["AD_CLASS"].notna()].copy()

In [ ]:
#keep only people on exactly one antidepressant class
ad_counts = df_ad.groupby("SEQN")["AD_CLASS"].nunique().reset_index()

#filters to keep only rows where ad_class = 1
ad_counts = ad_counts[ad_counts["AD_CLASS"] == 1]

#extracts participant IDs
single_class_ids = ad_counts["SEQN"]

#create subset for only participants with single class ID
df_ad_single = df_ad[df_ad["SEQN"].isin(single_class_ids)].copy()

#select only unique combos of participant ID/treatment class to remove duplicates
df_treatment = df_ad_single[["SEQN", "AD_CLASS"]].drop_duplicates()


In [ ]:
#drop tca and map treatment codes
df_treatment = df_treatment[df_treatment["AD_CLASS"] != "TCA"].copy()

#map treatment class to value 0-2
treat_map = {
    "ATYPICAL": 0,
    "SNRI": 1,
    "SSRI": 2
}

#convert treatment class labels into numeric codes & put in treatment column
df_treatment["treatment"] = df_treatment["AD_CLASS"].map(treat_map).astype(int)


#check
print(df_treatment["AD_CLASS"].value_counts())

In [ ]:
# rebuild data from survey files only
data = demo.merge(alq, on="SEQN", how="inner")
data = data.merge(dpq, on="SEQN", how="left")
data = data.merge(slq, on="SEQN", how="left")
data = data.merge(duq, on="SEQN", how="left")
data = data.merge(mcq, on="SEQN", how="left")
data = data.merge(smq, on="SEQN", how="left")

# now merge the clean treatment table
data["SEQN"] = data["SEQN"].astype(int)

#convert ID column to integer for merging
df_treatment["SEQN"] = df_treatment["SEQN"].astype(int)

#merge treatment info into data by ID keeping only rows in both datasets
#this elimates any participants not using 1 of the 3 antidpressants
data = data.merge(df_treatment[["SEQN", "AD_CLASS", "treatment"]], on="SEQN", how="inner")

print(data.shape)

In [ ]:
#now rebuild outcome and features
phq_cols = [
    "DPQ010","DPQ020","DPQ030","DPQ040",
    "DPQ050","DPQ060","DPQ070","DPQ080","DPQ090"
]

#redo the depression score indicator
data["PHQ9_TOTAL"] = data[phq_cols].sum(axis=1)
data["DEPRESSED"] = (data["PHQ9_TOTAL"] >= 10).astype(int)

data["short_sleep"] = (data["SLD012"] < 6).astype(int)
data["sleep_problem"] = (data["SLQ040"] >= 2).astype(int)
data["sleep_med"] = (data["SLQ120"] == 1).astype(int)

# Alcohol use indicator
data["alcohol_user"] = (data["ALQ101"] == 1).astype(int)

# Heavy drinking indicator (approximate threshold)
data["heavy_drinker"] = (data["ALQ130"] >= 4).astype(int)

# Frequent drinking indicator
data["frequent_drinker"] = (data["ALQ120Q"] >= 3).astype(int)

#assign feature columns
X_cols = [
"RIDAGEYR",
"RIAGENDR",
"RIDRETH1",
"INDFMPIR",
"SMQ020",
"MCQ010",
"MCQ160A",
"MCQ160F",
"short_sleep",
"sleep_problem",
"sleep_med",
"alcohol_user",
"heavy_drinker",
"frequent_drinker"
]

#identify missing codes
missing_codes = [7, 9, 77, 99, 777, 999]

#replace missing codes with nan
model_df = data[["DEPRESSED", "treatment"] + X_cols].replace(missing_codes, np.nan).dropna().copy()

#check
print(model_df["DEPRESSED"].mean())
print(model_df["treatment"].value_counts())
print(model_df["AD_CLASS"].value_counts() if "AD_CLASS" in model_df.columns else "AD_CLASS not in model_df")

In [ ]:
#check
print(df_treatment["AD_CLASS"].value_counts())
print(data.shape)
print(model_df.shape)

In [ ]:
#check
print(model_df.shape)
print(model_df["DEPRESSED"].mean())
print(model_df["treatment"].value_counts())

In [ ]:
#create a summary table by treatment group to show number of observations
#and the avg depression rate for each group
raw_summary = model_df.groupby("treatment")["DEPRESSED"].agg(
    N="count",
    Depressed_Rate="mean"
)

print(raw_summary)


In [ ]:
#map labels to get another table with antidepressant class name instead of integer
label_map = {0: "ATYPICAL", 1: "SNRI", 2: "SSRI"}
raw_summary.index = raw_summary.index.map(label_map)

print(raw_summary)

In [ ]:
#rebuild the feature, treatment and outcome variables
X = model_df[X_cols]
T = model_df["treatment"]
y = model_df["DEPRESSED"]

#create training set
X_train, X_temp, T_train, T_temp, y_train, y_temp = train_test_split(
    X, T, y,
    test_size=0.30,
    stratify=T,
    random_state=22
)

#create validation and test set
X_val, X_test, T_val, T_test, y_val, y_test = train_test_split(
    X_temp, T_temp, y_temp,
    test_size=0.50,
    stratify=T_temp,
    random_state=22
)

#show numbers to check split
print(len(X_train), len(X_val), len(X_test))

In [ ]:
#prepare causal arrays (econml expects numpy arrays)
X_train_causal = X_train.values
T_train_causal = T_train.values
Y_train_causal = y_train.values

#convert the validation df into a NumPy array for causal model
X_val_causal = X_val.values

In [ ]:
#define the models used inside the DR-Learner
#going to use XGBoost for both outcome and propensity models
model_regression = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=22,
    eval_metric="logloss"
)

#create propensity model
model_propensity = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=22,
    eval_metric="mlogloss"
)

In [ ]:
#train the DRLearner
#this will estimate hte treatment effects or how each antidepressant changes
#depression probability for each individual
dr = DRLearner(
    model_regression=model_regression,
    model_propensity=model_propensity,
    discrete_outcome=True,
    categories=[0,1,2],
    random_state=22
)

#fit the model
dr.fit(
    Y_train_causal,
    T_train_causal,
    X=X_train_causal
)

In [ ]:
#compute the treatment effects for validation data
marginal = dr.const_marginal_effect(X_val_causal)

#extract slice of marginal array
marginal2 = marginal[:,0,:]

print(marginal2.shape)
#expected is 256, 2 because 3 treatments, 1 baseline, so 2 estimated constrasts

In [ ]:
#create a copy of the training feature set so original data not modified
X_train_outcome = X_train.copy()

#add the treatment variable to the training features as an additional feature
X_train_outcome["treatment"] = T_train.values

#convert the training outcome variable to a NumPy array
y_train_outcome = y_train.values

#crate a copy of the validation set
X_val_outcome = X_val.copy()

#add the treatment variable to the validation features as an additional feature
X_val_outcome["treatment"] = T_val.values

#convert the validation outcome variable to a NumPy array
y_val_outcome = y_val.values

In [ ]:
#fit a standalone outcome model
outcome_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=22,
    eval_metric="logloss"
)

#fit the model
outcome_model.fit(X_train_outcome, y_train_outcome)

In [ ]:
#predict risk under each treatment
#create 3 copies of the validation set and set treatment explicitly
X_val_t0 = X_val.copy()
X_val_t1 = X_val.copy()
X_val_t2 = X_val.copy()

#create counterfactual estimates
X_val_t0["treatment"] = 0   # ATYPICAL
X_val_t1["treatment"] = 1   # SNRI
X_val_t2["treatment"] = 2   # SSRI

#evaluate counterfactual estimates
risk_ATYPICAL = outcome_model.predict_proba(X_val_t0)[:, 1]
risk_SNRI     = outcome_model.predict_proba(X_val_t1)[:, 1]
risk_SSRI     = outcome_model.predict_proba(X_val_t2)[:, 1]

In [ ]:
#build risk table
risk_df = pd.DataFrame({
    "risk_ATYPICAL": risk_ATYPICAL,
    "risk_SNRI": risk_SNRI,
    "risk_SSRI": risk_SSRI
})

#show
print(risk_df.describe())

In [ ]:
#recommended treatment
risk_df["recommended_treatment"] = risk_df.idxmin(axis=1)

#show
print(risk_df["recommended_treatment"].value_counts(normalize=True))

In [ ]:
#validation AUROC and Brier
val_probs = outcome_model.predict_proba(X_val_outcome)[:, 1]

#show
print("Validation AUROC:", roc_auc_score(y_val_outcome, val_probs))
print("Validation Brier:", brier_score_loss(y_val_outcome, val_probs))

In [ ]:
# Risk under observed treatment
observed_risk = outcome_model.predict_proba(X_val_outcome)[:, 1]

# Risk under recommended treatment
recommended_risk = []

for i in range(len(X_val)):
    rec = risk_df["recommended_treatment"].iloc[i]
    recommended_risk.append(risk_df.loc[i, rec])

recommended_risk = np.array(recommended_risk)

#show
print("Mean observed risk:", observed_risk.mean())
print("Mean recommended risk:", recommended_risk.mean())
print("Absolute risk reduction:", observed_risk.mean() - recommended_risk.mean())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#create a heatmap plot for treatment distribution by sex
plot_df = X_val.copy()
plot_df["recommended"] = risk_df["recommended_treatment"]

heat = pd.crosstab(
    plot_df["RIAGENDR"],
    plot_df["recommended"],
    normalize="index"
)

sns.heatmap(heat, annot=True, cmap="Blues")

plt.title("Recommended Treatment Distribution by Sex")
plt.xlabel("Recommended Treatment")
plt.ylabel("Sex (1=Male, 2=Female)")
plt.show()

In [ ]:
#create a heatmap plot for treatment recommendations by age
plot_df["age_group"] = pd.cut(plot_df["RIDAGEYR"], bins=[18,30,45,60,80])

heat = pd.crosstab(
    plot_df["age_group"],
    plot_df["recommended"],
    normalize="index"
)

sns.heatmap(heat, annot=True, cmap="Blues")
plt.title("Treatment Recommendations by Age Group")
plt.show()

In [ ]:
pip install shap

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#convert the validation features into XGBoost DMatrix for evaluation
dm_val = xgb.DMatrix(X_val_outcome)

# SHAP values from XGBoost directly
shap_vals = outcome_model.get_booster().predict(dm_val, pred_contribs=True)

# Drop the last column (the bias term)
shap_df = pd.DataFrame(
    shap_vals[:, :-1],
    columns=X_val_outcome.columns
)

# Mean absolute SHAP importance
importance = shap_df.abs().mean().sort_values(ascending=False)

print(importance)

# Plot
importance.sort_values().plot(kind="barh", figsize=(8,6))
plt.title("SHAP Feature Importance")
plt.xlabel("Mean |SHAP value|")
plt.show()

In [ ]:
i = 0  # choose a patient row

#selects SHAP values for a single patient row and sorts features by importance
patient_shap = shap_df.iloc[i].sort_values(key=np.abs, ascending=False)

#show
print(patient_shap)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

#create boxplot to show predicted depression risk under each treatment
risk_long = risk_df.melt(
    value_vars=["risk_ATYPICAL", "risk_SNRI", "risk_SSRI"],
    var_name="treatment",
    value_name="risk"
)

plt.figure(figsize=(8,6))
sns.boxplot(x="treatment", y="risk", data=risk_long)
plt.title("Predicted Depression Risk Under Each Treatment")
plt.ylabel("Predicted Risk")
plt.xlabel("Treatment")
plt.show()

In [ ]:
# Build holdout dataframe for the standalone outcome model
X_test_outcome = X_test.copy()
X_test_outcome["treatment"] = T_test.values
y_test_outcome = y_test.values

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss

#evaluate
test_probs = outcome_model.predict_proba(X_test_outcome)[:, 1]

#show
print("Test AUROC:", roc_auc_score(y_test_outcome, test_probs))
print("Test Brier:", brier_score_loss(y_test_outcome, test_probs))

In [ ]:
#create a copy of the test features for Atypical treatment scenario
X_test_t0 = X_test.copy()

#create a copy of the test features for SNRI treatment scenario
X_test_t1 = X_test.copy()

#create a copy of the test features for SSRI treatment scenario
X_test_t2 = X_test.copy()

#sets treatment to atypical for all patients to simulate that scenario
X_test_t0["treatment"] = 0   # ATYPICAL

#sets treatment to SNRI for all patients to simulate that scenario
X_test_t1["treatment"] = 1   # SNRI

#sets treatment to SSRI for all patients to simulate that scenario
X_test_t2["treatment"] = 2   # SSRI


#evaluates each if they received atypical - SSRI
risk_ATYPICAL_test = outcome_model.predict_proba(X_test_t0)[:, 1]
risk_SNRI_test = outcome_model.predict_proba(X_test_t1)[:, 1]
risk_SSRI_test = outcome_model.predict_proba(X_test_t2)[:, 1]

In [ ]:
#create a risk matrix
risk_test_df = pd.DataFrame({
    "risk_ATYPICAL": risk_ATYPICAL_test,
    "risk_SNRI": risk_SNRI_test,
    "risk_SSRI": risk_SSRI_test
})

print(risk_test_df.describe())

In [ ]:
#creates a new column assigning each patient the treatment with lowest predicted risk
risk_test_df["recommended_treatment"] = risk_test_df.idxmin(axis=1)

print(risk_test_df["recommended_treatment"].value_counts(normalize=True))

In [ ]:
# Predicted risk under the treatment the patient actually received
observed_risk_test = outcome_model.predict_proba(X_test_outcome)[:, 1]

# Predicted risk under the model-recommended treatment
recommended_risk_test = []

for i in range(len(X_test)):
    rec = risk_test_df["recommended_treatment"].iloc[i]
    recommended_risk_test.append(risk_test_df.loc[i, rec])

import numpy as np
recommended_risk_test = np.array(recommended_risk_test)

print("Mean observed risk (test):", observed_risk_test.mean())
print("Mean recommended risk (test):", recommended_risk_test.mean())
print("Absolute risk reduction (test):", observed_risk_test.mean() - recommended_risk_test.mean())

In [ ]:
#Treatment Recommendation Distribution
#shows how often each antidepressant is recommended
rec = risk_test_df["recommended_treatment"].value_counts(normalize=True)

plt.figure(figsize=(6,5))
rec.plot(kind="bar")

plt.title("Distribution of Recommended Antidepressant Treatments")
plt.ylabel("Proportion of Patients")
plt.xlabel("Treatment")

plt.show()

In [ ]:
#SHAP Feature Importance
#Global feature importance using SHAP values
#Socioeconomic status, age, comorbidities, and behavioral factors were the
#strongest predictors of depression risk
importance.sort_values().plot(
    kind="barh",
    figsize=(8,6)
)

plt.title("SHAP Feature Importance")
plt.xlabel("Mean |SHAP Value|")
plt.show()

In [ ]:
#predicted risk by treatment
#Distribution of predicted depression risk under each antidepressant class
#SSRIs had the lowest average predicted risk across the test population
risk_long = risk_test_df.melt(
    value_vars=["risk_ATYPICAL","risk_SNRI","risk_SSRI"],
    var_name="treatment",
    value_name="risk"
)

plt.figure(figsize=(7,6))

sns.boxplot(
    x="treatment",
    y="risk",
    data=risk_long
)

plt.title("Predicted Depression Risk Under Each Treatment")
plt.ylabel("Predicted Risk")
plt.xlabel("Treatment")

plt.show()

In [ ]:
#confidence interval for policy improvement
#should estimate a 95% confidence interval using bootstrap resampling
n_boot = 2000

#initialize an empty list to store the bootstrap estimates
policy_effects = []

#repeat the bootstrap n_boot times
for _ in range(n_boot):

    #generate a bootstrap sample of indices by randomly sampling with replacement
    idx = np.random.choice(
        len(observed_risk_test),
        len(observed_risk_test),
        replace=True
    )

    #compute the difference in average risk between observed treatment and
    #recommended treatment for the sampled data
    effect = (
        observed_risk_test[idx].mean()
        - recommended_risk_test[idx].mean()
    )

    #store the computed policy effect from this bootstrap iteration
    policy_effects.append(effect)

#convert the list of bootstrap effects into a NumPy array
policy_effects = np.array(policy_effects)

#calculate the lower bound of the 95% confidence interval (2.5 percentile)
ci_lower = np.percentile(policy_effects, 2.5)

#calculate the upper bound of the 95% confidence interval (97.5 percentile)
ci_upper = np.percentile(policy_effects, 97.5)

#print the estimate average improvement from using the recommended treatment
print("Policy improvement:", observed_risk_test.mean() - recommended_risk_test.mean())

#print the 95% confidence interval for the policy improvment estimate
print("95% CI:", ci_lower, ci_upper)

In [ ]:
#treatment benefit curve
# individual treatment benefit
benefit = observed_risk_test - recommended_risk_test

# sort largest benefit first
benefit_sorted = np.sort(benefit)[::-1]

# cumulative mean benefit
cum_benefit = np.cumsum(benefit_sorted) / np.arange(1, len(benefit_sorted)+1)

plt.figure(figsize=(7,6))

plt.plot(cum_benefit)

plt.axhline(
    benefit.mean(),
    linestyle="--"
)

plt.xlabel("Patients ranked by treatment benefit")
plt.ylabel("Average risk reduction")

plt.title("Treatment Benefit Curve")

plt.show()

#Figure 5. Treatment benefit curve.
#Patients were ranked by predicted improvement under the personalized treatment
#policy. The curve illustrates heterogeneity in treatment benefit, with a subset
#of patients predicted to experience larger reductions in depression risk.

In [ ]:
from sklearn.calibration import calibration_curve

# get predicted probabilities from test set
test_probs = outcome_model.predict_proba(X_test_outcome)[:,1]

# compute calibration curve
prob_true, prob_pred = calibration_curve(
    y_test_outcome,
    test_probs,
    n_bins=6
)

plt.figure(figsize=(6,6))

# perfect calibration line
plt.plot([0,1],[0,1], linestyle="--")

# model calibration
plt.plot(prob_pred, prob_true, marker="o")

plt.xlabel("Predicted Probability")
plt.ylabel("Observed Frequency")

plt.title("Calibration Plot (Test Set)")

plt.show()

In [ ]:
#MODEL PERFORMANCE SUMMARY
table = pd.DataFrame({
    "Metric": [
        "AUROC",
        "Brier Score",
        "Mean Observed Risk",
        "Mean Recommended Risk",
        "Absolute Risk Reduction",
        "95% CI Lower",
        "95% CI Upper"
    ],
    "Validation": [
        0.6276,
        0.1758,
        None,
        None,
        None,
        None,
        None
    ],
    "Test": [
        0.6599,
        0.1643,
        0.2387,
        0.2160,
        0.0228,
        0.0180,
        0.0278
    ]
})

print(table)

In [ ]:
# Create DMatrix from validation data
dm_val = xgb.DMatrix(X_val_outcome)

# Get SHAP contributions from XGBoost directly
shap_vals = outcome_model.get_booster().predict(dm_val, pred_contribs=True)

# Drop the last column (bias term)
shap_df = pd.DataFrame(
    shap_vals[:, :-1],
    columns=X_val_outcome.columns
)

# Mean absolute SHAP importance
importance = shap_df.abs().mean().sort_values(ascending=False)

print(importance)

In [ ]:
#create an individual shap plot
patient_idx = 0

patient_shap = shap_df.iloc[patient_idx].sort_values(key=np.abs, ascending=True)

plt.figure(figsize=(8, 6))

patient_shap.plot(
    kind="barh",
    color=["#0B7A3E" if v > 0 else "#B8A16A" for v in patient_shap]
)

plt.title("Individual SHAP Explanation for Patient 0", color="#1B4F8C", fontsize=14)
plt.xlabel("SHAP Value", color="#1B4F8C")
plt.ylabel("Feature", color="#1B4F8C")

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#B8A16A")
    spine.set_linewidth(1.5)

ax.tick_params(colors="#1B4F8C")

plt.tight_layout()
plt.show()

In [ ]:
sns.set_style("whitegrid")

# Main paper color from Overleaf
PAPER_BLUE = "#004C97"

# Variations of the same blue
BLUE_DARK   = "#003B75"
BLUE_MAIN   = "#004C97"
BLUE_MEDIUM = "#2E6FAF"
BLUE_LIGHT  = "#6FA3D2"
BLUE_PALE   = "#BFD7EA"

BLUE_PALETTE_3 = [BLUE_DARK, BLUE_MAIN, BLUE_MEDIUM]
BLUE_PALETTE_4 = [BLUE_DARK, BLUE_MAIN, BLUE_MEDIUM, BLUE_LIGHT]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": BLUE_MAIN,
    "axes.labelcolor": BLUE_DARK,
    "axes.titlecolor": BLUE_DARK,
    "xtick.color": BLUE_DARK,
    "ytick.color": BLUE_DARK,
    "text.color": BLUE_DARK,
    "axes.titleweight": "bold",
    "axes.linewidth": 1.2,
    "grid.color": BLUE_PALE,
    "grid.alpha": 0.6,
    "font.size": 12,
    "legend.frameon": False
})

label_map = {0: "Atypical", 1: "SNRI", 2: "SSRI"}

In [ ]:
#create a barplot showing treatment class distribution
treat_counts = model_df["treatment"].value_counts().sort_index()
treat_counts.index = treat_counts.index.map(label_map)

plt.figure(figsize=(7,5))
ax = sns.barplot(
    x=treat_counts.index,
    y=treat_counts.values,
    hue=treat_counts.index,
    palette=BLUE_PALETTE_3,
    legend=False
)

for i, v in enumerate(treat_counts.values):
    ax.text(i, v + 20, f"{v}", ha="center", fontweight="bold", color=BLUE_DARK)

plt.title("Treatment Class Distribution in Final Analytic Dataset")
plt.xlabel("Antidepressant Class")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.savefig("treatment_distribution_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#create a barplot showing observed depression rate by treatment class
raw_rates = model_df.groupby("treatment")["DEPRESSED"].mean().sort_index()
raw_rates.index = raw_rates.index.map(label_map)

plt.figure(figsize=(7,5))
ax = sns.barplot(
    x=raw_rates.index,
    y=raw_rates.values,
    hue=raw_rates.index,
    palette=BLUE_PALETTE_3,
    legend=False
)

for i, v in enumerate(raw_rates.values):
    ax.text(i, v + 0.008, f"{v:.1%}", ha="center", fontweight="bold", color=BLUE_DARK)

plt.title("Observed Depression Rate by Treatment Class")
plt.xlabel("Antidepressant Class")
plt.ylabel("Observed Depression Rate")
plt.ylim(0, max(raw_rates.values) + 0.06)
plt.tight_layout()
plt.savefig("observed_depression_rate_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#create a barplot showing adjusted predicted depressed risk by treatment class
adjusted_risks = pd.Series({
    "Atypical": 0.252,
    "SNRI": 0.250,
    "SSRI": 0.217
})

plt.figure(figsize=(7,5))
ax = sns.barplot(
    x=adjusted_risks.index,
    y=adjusted_risks.values,
    hue=adjusted_risks.index,
    palette=BLUE_PALETTE_3,
    legend=False
)

for i, v in enumerate(adjusted_risks.values):
    ax.text(i, v + 0.008, f"{v:.1%}", ha="center", fontweight="bold", color=BLUE_DARK)

plt.title("Adjusted Predicted Depression Risk by Treatment Class")
plt.xlabel("Antidepressant Class")
plt.ylabel("Predicted Depression Risk")
plt.ylim(0, max(adjusted_risks.values) + 0.06)
plt.tight_layout()
plt.savefig("adjusted_predicted_risk_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#create a distribution plot to show distribution of recommended treatment classes
rec_dist = pd.Series({
    "SSRI": 0.699219,
    "Atypical": 0.269531,
    "SNRI": 0.031250
})

plt.figure(figsize=(7,5))
ax = sns.barplot(
    x=rec_dist.index,
    y=rec_dist.values,
    hue=rec_dist.index,
    palette=[BLUE_MAIN, BLUE_MEDIUM, BLUE_LIGHT],
    legend=False
)

for i, v in enumerate(rec_dist.values):
    ax.text(i, v + 0.01, f"{v:.1%}", ha="center", fontweight="bold", color=BLUE_DARK)

plt.title("Distribution of Recommended Treatment Classes")
plt.xlabel("Recommended Treatment")
plt.ylabel("Proportion of Patients")
plt.ylim(0, max(rec_dist.values) + 0.08)
plt.tight_layout()
plt.savefig("recommended_treatment_distribution_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#create barplot to show differences in evaluation scores between validation and test
#sets
perf_df = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "AUROC": [0.6276, 0.6599],
    "Brier Score": [0.1758, 0.1643]
})

# AUROC
plt.figure(figsize=(6,5))
ax = sns.barplot(
    data=perf_df,
    x="Dataset",
    y="AUROC",
    hue="Dataset",
    palette=[BLUE_MEDIUM, BLUE_MAIN],
    legend=False
)

for i, v in enumerate(perf_df["AUROC"]):
    ax.text(i, v + 0.01, f"{v:.4f}", ha="center", fontweight="bold", color=BLUE_DARK)

plt.title("AUROC on Validation and Test Sets")
plt.ylabel("AUROC")
plt.xlabel("")
plt.ylim(0, 0.8)
plt.tight_layout()
plt.savefig("auroc_validation_test_blue.png", dpi=300, bbox_inches="tight")
plt.show()

# Brier Score
plt.figure(figsize=(6,5))
ax = sns.barplot(
    data=perf_df,
    x="Dataset",
    y="Brier Score",
    hue="Dataset",
    palette=[BLUE_MEDIUM, BLUE_MAIN],
    legend=False
)

for i, v in enumerate(perf_df["Brier Score"]):
    ax.text(i, v + 0.005, f"{v:.4f}", ha="center", fontweight="bold", color=BLUE_DARK)

plt.title("Brier Score on Validation and Test Sets")
plt.ylabel("Brier Score")
plt.xlabel("")
plt.ylim(0, 0.25)
plt.tight_layout()
plt.savefig("brier_validation_test_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#create a barplot to show observed vs. model recommended depression risk
policy_df = pd.DataFrame({
    "Metric": ["Observed Risk", "Recommended Risk"],
    "Risk": [0.2387, 0.2160]
})

plt.figure(figsize=(6,5))
ax = sns.barplot(
    data=policy_df,
    x="Metric",
    y="Risk",
    hue="Metric",
    palette=[BLUE_MEDIUM, BLUE_MAIN],
    legend=False
)

for i, v in enumerate(policy_df["Risk"]):
    ax.text(i, v + 0.006, f"{v:.1%}", ha="center", fontweight="bold", color=BLUE_DARK)

plt.title("Observed vs Model-Recommended Depression Risk")
plt.ylabel("Mean Depression Risk")
plt.xlabel("")
plt.ylim(0, 0.30)
plt.tight_layout()
plt.savefig("observed_vs_recommended_risk_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#policy improvement with confidence interval plot
arr = 0.0228
ci_lower = 0.0180
ci_upper = 0.0278

plt.figure(figsize=(6,4))
plt.errorbar(
    x=[arr],
    y=[0],
    xerr=[[arr - ci_lower], [ci_upper - arr]],
    fmt='o',
    markersize=8,
    capsize=6,
    color=BLUE_MAIN,
    ecolor=BLUE_MEDIUM,
    elinewidth=2
)

plt.axvline(0, linestyle="--", linewidth=1, color=BLUE_LIGHT)
plt.yticks([])
plt.xlabel("Absolute Risk Reduction")
plt.title("Policy Improvement with 95% Confidence Interval")
plt.xlim(0, 0.035)
plt.tight_layout()
plt.savefig("policy_improvement_ci_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#barplot to show observed risk vs. adjusted prediction depression risk
compare_df = pd.DataFrame({
    "Treatment": ["Atypical", "SNRI", "SSRI"],
    "Raw Observed Risk": [0.246334, 0.275862, 0.221719],
    "Adjusted Predicted Risk": [0.252, 0.250, 0.217]
})

compare_long = compare_df.melt(
    id_vars="Treatment",
    var_name="Risk Type",
    value_name="Risk"
)

plt.figure(figsize=(8,5))
ax = sns.barplot(
    data=compare_long,
    x="Treatment",
    y="Risk",
    hue="Risk Type",
    palette=[BLUE_LIGHT, BLUE_MAIN]
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", labels=[f"{v*100:.1f}%" for v in container.datavalues], padding=3, color=BLUE_DARK)

plt.title("Raw Observed vs Adjusted Predicted Depression Risk")
plt.xlabel("Antidepressant Class")
plt.ylabel("Depression Risk")
plt.ylim(0, 0.35)
plt.legend(title="")
plt.tight_layout()
plt.savefig("raw_vs_adjusted_risk_blue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#change visualizations color scheme to same blue as in overleaf paper
sns.set_style("whitegrid")

# --- Your color system ---
PAPER_BLUE = "#004C97"

BLUE_DARK   = "#003B75"
BLUE_MAIN   = "#004C97"
BLUE_MEDIUM = "#2E6FAF"
BLUE_LIGHT  = "#6FA3D2"
BLUE_PALE   = "#BFD7EA"

BLUE_PALETTE_3 = [BLUE_DARK, BLUE_MAIN, BLUE_MEDIUM]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",   # <-- black outline
    "axes.labelcolor": BLUE_DARK,
    "axes.titlecolor": BLUE_DARK,
    "xtick.color": BLUE_DARK,
    "ytick.color": BLUE_DARK,
    "text.color": BLUE_DARK,
    "axes.titleweight": "bold",
    "axes.linewidth": 1.2,
    "grid.color": BLUE_PALE,
    "grid.alpha": 0.6,
    "font.size": 12,
    "legend.frameon": False
})

# --- Reshape data ---
risk_long = risk_test_df.melt(
    value_vars=["risk_ATYPICAL","risk_SNRI","risk_SSRI"],
    var_name="treatment",
    value_name="risk"
)

# Clean labels
risk_long["treatment"] = risk_long["treatment"].replace({
    "risk_ATYPICAL": "Atypical",
    "risk_SNRI": "SNRI",
    "risk_SSRI": "SSRI"
})

In [ ]:
#boxplot for predicted depression risk under each treatment
plt.figure(figsize=(7,6))

ax = sns.boxplot(
    data=risk_long,
    x="treatment",
    y="risk",
    palette=BLUE_PALETTE_3,
    width=0.6,
    linewidth=1.5
)

# --- Force black outlines on EVERYTHING ---
for artist in ax.artists:
    artist.set_edgecolor("black")

for line in ax.lines:
    line.set_color("black")

# --- Labels ---
plt.title("Predicted Depression Risk Under Each Treatment")
plt.ylabel("Predicted Risk")
plt.xlabel("Treatment")

plt.tight_layout()

plt.savefig("Predicted_Depression_Boxplot.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
#create treatment benefit curve plot
benefit = observed_risk_test - recommended_risk_test

benefit_sorted = np.sort(benefit)[::-1]

cum_benefit = np.cumsum(benefit_sorted) / np.arange(1, len(benefit_sorted)+1)

# --- Plot ---
plt.figure(figsize=(7,6))

plt.plot(
    cum_benefit,
    color=BLUE_MAIN,
    linewidth=2.5,
    label="Cumulative Mean Benefit"
)

# --- Average benefit line ---
plt.axhline(
    benefit.mean(),
    color=BLUE_DARK,
    linestyle="--",
    linewidth=2,
    label="Average Benefit"
)

# --- Labels ---
plt.xlabel("Patients Ranked by Treatment Benefit")
plt.ylabel("Average Risk Reduction")
plt.title("Treatment Benefit Curve")

# --- Optional: nicer x-axis scaling ---
plt.xlim(0, len(cum_benefit))

# --- Legend ---
plt.legend()

# --- Clean look ---
sns.despine()

plt.tight_layout()

plt.savefig("Treatment_Benefit_Curve.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
#create calibration plot
# --- Plot ---
plt.figure(figsize=(6,6))

# Perfect calibration line
plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color=BLUE_MAIN,
    linewidth=1.8,
    label="Perfect Calibration"
)

# Model calibration curve
plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    markersize=7,
    linewidth=2.2,
    color=BLUE_DARK,
    markerfacecolor=BLUE_MEDIUM,
    markeredgecolor="black",
    label="Model"
)

plt.xlabel("Predicted Probability")
plt.ylabel("Observed Frequency")
plt.title("Calibration Plot (Test Set)")

plt.xlim(0, 1)
plt.ylim(0, 1.05)

plt.legend()

sns.despine()

plt.tight_layout()

plt.savefig("Calibration_Plot.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
#create shap feature importance plot
plt.figure(figsize=(8, 6))

# --- Gradient blue palette for bars ---
colors = [BLUE_DARK, BLUE_MAIN, BLUE_MEDIUM, BLUE_LIGHT, BLUE_PALE]

# Expand palette to match number of features
palette = colors * (len(importance) // len(colors) + 1)

# --- Plot ---
ax = importance.sort_values().plot(
    kind="barh",
    color=palette[:len(importance)],
    edgecolor="black",     # black bar outlines
    linewidth=1.0
)

# --- Labels ---
plt.title("SHAP Feature Importance")
plt.xlabel("Mean |SHAP Value|")
plt.ylabel("Feature")

# --- Clean spines (keep black border) ---
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.2)

# --- Optional: emphasize top features visually ---
# (makes figure look more "publication grade")
for i, bar in enumerate(ax.patches):
    if i >= len(ax.patches) - 3:  # top 3 features
        bar.set_color(BLUE_DARK)

sns.despine(left=False, bottom=False)

plt.tight_layout()

plt.savefig("SHAP_Feature_Importance.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import os
print(os.getcwd())

In [ ]:
#create roc curve plot 
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# predicted probabilities
y_probs = outcome_model.predict_proba(X_test_outcome)[:, 1]

# ROC pieces
fpr, tpr, thresholds = roc_curve(y_test_outcome, y_probs)
roc_auc = roc_auc_score(y_test_outcome, y_probs)

plt.figure(figsize=(6, 6))

plt.plot(
    fpr, tpr,
    color=BLUE_MAIN,
    linewidth=2.5,
    label=f"Model (AUROC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="black",
    linewidth=1.5,
    label="Random"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Test Set)")
plt.xlim(0, 1)
plt.ylim(0, 1)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.2)

plt.legend()
plt.tight_layout()
plt.savefig("roc_curve.pdf", bbox_inches="tight")
plt.show()

In [ ]:
print("After antidepressant filter:", df_treatment.shape[0])